# 📈 KPI Profiling & Correlation Analysis
## Mérida Urban Intelligence | Sunchos Team — Data Analysis (Bianca)

**Goal.** Audit the territorial KPIs produced by the Data Warehouse and analyse at least three
relationships between demographic, economic and public-safety indicators using **Pearson** and
**Spearman** correlation.

**Data source rule.** Following the project specification, every number in this notebook is read
from the PostgreSQL/PostGIS warehouse view `v_kpis_territoriales` (`sql/03_views.sql`) and the fact
tables — never from raw files. A CSV fallback (`data/processed/merida_kpis_real_baseline.csv`) exists only
so the notebook can be opened offline; results for the report must come from the warehouse.

**Outputs**
| File | Purpose |
| :--- | :--- |
| `outputs/figures/corr_*.png`, `outputs/figures/dist_*.png` | Figures for the technical report |
| `outputs/analytics/correlation_results.csv` | Hypothesis-test table (ρ, r, p-values, n) |
| `outputs/analytics/analytics_summary.json` | Data feed for the ApexCharts dashboard components |

KPI definitions and source variables: `docs/kpi_formulas.md`.

## 0. Setup

In [ ]:
import json
import os
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)

# Locate repository root (works when launched from repo root or from notebooks/)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

DATABASE_URL = os.getenv("DATABASE_URL", "")
FIG_DIR = ROOT / "outputs" / "figures"
ANALYTICS_DIR = ROOT / "outputs" / "analytics"
FRONTEND_DATA_DIR = ROOT / "src" / "frontend" / "public" / "data"
for d in (FIG_DIR, ANALYTICS_DIR, FRONTEND_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

MIN_POPULATION = 100   # analytic sample rule for per-capita KPIs (see docs/kpi_formulas.md)
ALPHA = 0.05

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "savefig.bbox": "tight"})
print(f"Repository root: {ROOT}")
print(f"Warehouse connection configured: {bool(DATABASE_URL)}")

## 1. Load KPIs from the Data Warehouse

The main query reads the consolidated KPI view at AGEB grain (one row per `cvegeo`) and adds the
raw demographic counts needed for derived indicators. Two extra queries feed the categorical charts:
establishments by SCIAN sector and crime incidents by category and time.

In [ ]:
KPI_SQL = """
SELECT v.cvegeo, v.area_km2,
       v.poblacion_total, v.densidad_poblacion_km2, v.tasa_pea_porcentaje,
       v.poblacion_0_14, v.poblacion_15_64, v.poblacion_65_mas,
       v.total_negocios, v.densidad_negocios_km2, v.negocios_por_mil_hab,
       v.densidad_comercio_km2, v.densidad_servicios_km2, v.actividad_economica_dominante,
       v.total_delitos, v.tasa_delictiva_por_mil_hab, v.ratio_delito_por_negocio,
       fd.poblacion_pea, fd.poblacion_pnea, fd.total_viviendas
FROM v_kpis_territoriales v
LEFT JOIN fact_demografia fd ON fd.cvegeo = v.cvegeo
ORDER BY v.cvegeo;
"""

SECTOR_SQL = """
SELECT dae.sector_codigo, dae.categoria_macro, COUNT(*) AS establecimientos
FROM fact_negocios fn
JOIN dim_actividad_economica dae ON dae.scian_id = fn.scian_id
GROUP BY dae.sector_codigo, dae.categoria_macro
ORDER BY establecimientos DESC;
"""

CRIME_SQL = """
SELECT fc.categoria_delito,
       COALESCE(fc.periodo_dia, 'Unknown') AS periodo_dia,
       dt.dia_semana, dt.anio, dt.mes,
       COUNT(*) AS incidentes
FROM fact_crimen fc
LEFT JOIN dim_tiempo dt ON dt.tiempo_id = fc.tiempo_id
GROUP BY 1, 2, 3, 4, 5;
"""

DATA_SOURCE = None
df_sector = pd.DataFrame(columns=["sector_codigo", "categoria_macro", "establecimientos"])
df_crime_t = pd.DataFrame(columns=["categoria_delito", "periodo_dia", "dia_semana", "anio", "mes", "incidentes"])

if DATABASE_URL:
    from sqlalchemy import create_engine, text
    db_url = DATABASE_URL
    # requirements.txt ships psycopg2; pin the driver so newer SQLAlchemy versions don't look for psycopg 3
    if db_url.startswith("postgres://"):
        db_url = "postgresql://" + db_url[len("postgres://"):]
    if db_url.startswith("postgresql://"):
        db_url = "postgresql+psycopg2://" + db_url[len("postgresql://"):]
    engine = create_engine(db_url, pool_pre_ping=True)
    with engine.connect() as conn:
        df = pd.read_sql(text(KPI_SQL), conn)
        df_sector = pd.read_sql(text(SECTOR_SQL), conn)
        df_crime_t = pd.read_sql(text(CRIME_SQL), conn)
    DATA_SOURCE = "PostgreSQL/PostGIS warehouse (v_kpis_territoriales)"
else:
    fallback = ROOT / "data" / "processed" / "merida_kpis_real_baseline.csv"
    if not fallback.exists():
        raise FileNotFoundError(
            "DATABASE_URL is not set and the offline CSV fallback was not found. "
            "Copy .env.example to .env and set DATABASE_URL to the Supabase connection string."
        )
    df = pd.read_csv(fallback, dtype={"cvegeo": str})
    DATA_SOURCE = f"OFFLINE FALLBACK: {fallback.name} (do not use for final results)"
    print("⚠️ Using the offline CSV fallback. Final results must be generated from the warehouse.")

# The offline CSV may not carry every column of the view: add missing ones as empty
EXPECTED = ["cvegeo", "area_km2", "poblacion_total", "densidad_poblacion_km2", "tasa_pea_porcentaje",
            "poblacion_0_14", "poblacion_15_64", "poblacion_65_mas", "total_negocios", "densidad_negocios_km2",
            "negocios_por_mil_hab", "densidad_comercio_km2", "densidad_servicios_km2",
            "actividad_economica_dominante", "total_delitos", "tasa_delictiva_por_mil_hab",
            "ratio_delito_por_negocio", "poblacion_pea", "poblacion_pnea", "total_viviendas"]
for col in EXPECTED:
    if col not in df.columns:
        df[col] = "" if col == "actividad_economica_dominante" else (0 if col.startswith(("total_", "poblacion_")) else np.nan)

# Numeric types (NUMERIC columns arrive as Decimal/object from PostgreSQL)
num_cols = [c for c in df.columns if c not in ("cvegeo", "actividad_economica_dominante")]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
df["cvegeo"] = df["cvegeo"].astype(str)
for frame in (df_sector, df_crime_t):
    for c in ("establecimientos", "incidentes", "anio", "mes"):
        if c in frame.columns:
            frame[c] = pd.to_numeric(frame[c], errors="coerce")

print(f"Source: {DATA_SOURCE}")
print(f"AGEB rows: {len(df):,} | columns: {df.shape[1]}")
df.head()

## 2. Data audit

Checks that the warehouse output is fit for analysis: grain, keys, missing values, zeros,
internal consistency of the census counts and availability of each thematic layer.

In [ ]:
audit = {}
audit["rows"] = len(df)
audit["duplicate_cvegeo"] = int(df["cvegeo"].duplicated().sum())
audit["cvegeo_length_ok"] = bool((df["cvegeo"].str.len() == 13).all())
audit["agebs_zero_population"] = int((df["poblacion_total"] == 0).sum())
audit["agebs_below_min_population"] = int((df["poblacion_total"] < MIN_POPULATION).sum())
audit["agebs_without_businesses"] = int((df["total_negocios"] == 0).sum())

# Census consistency: age groups must add up to (approximately) the total population
age_sum = df[["poblacion_0_14", "poblacion_15_64", "poblacion_65_mas"]].sum(axis=1)
age_gap = (df["poblacion_total"] - age_sum)
audit["age_groups_equal_total_0_14"] = int((df["poblacion_0_14"] == df["poblacion_total"]).sum())
audit["age_groups_valid"] = bool((df["poblacion_15_64"] > 0).mean() > 0.9)
audit["median_abs_age_gap"] = float(age_gap.abs().median())
audit["eap_rate_null_share"] = float(df["tasa_pea_porcentaje"].isna().mean())

CRIME_LOADED = bool(df["total_delitos"].sum() > 0)
BUSINESS_LOADED = bool(df["total_negocios"].sum() > 0)
AGE_VALID = audit["age_groups_valid"]
audit["business_layer_loaded"] = BUSINESS_LOADED
audit["crime_layer_loaded"] = CRIME_LOADED

for k, v in audit.items():
    print(f"{k:32s} {v}")

if not AGE_VALID:
    print("\n❌ Age groups look invalid (15–64 and 65+ are empty). The warehouse was loaded with the "
          "old census column mapping. Merge fix/bianca-census-age-columns, re-run the ETL and reload "
          "Supabase. Age-based indicators are excluded below.")
if not CRIME_LOADED:
    print("\nℹ️ fact_crimen is empty: crime KPIs and crime relationships (H4, H5) are skipped.")

In [ ]:
missing = df.isna().mean().mul(100).round(1).rename("% missing")
zeros = (df[num_cols] == 0).mean().mul(100).round(1).rename("% zero")
profile = pd.concat([df[num_cols].describe().T, missing, zeros], axis=1)
profile[["count", "mean", "50%", "std", "min", "max", "% missing", "% zero"]].round(2)

In [ ]:
dominant = (df.loc[df["total_negocios"] > 0, "actividad_economica_dominante"]
              .value_counts().head(10).rename("AGEBs where dominant"))
dominant.to_frame()

## 3. Derived indicators and analytic sample

* **Age shares** and **dependency ratio** turn counts into comparable proportions.
* **Crime per 100 establishments** rescales the view's crime-to-business ratio.
* Per-capita KPIs are unstable for tiny populations, so correlations use AGEBs with
  `poblacion_total ≥ 100` (documented rule).

In [ ]:
def safe_div(a, b, scale=1.0):
    b = b.replace(0, np.nan)
    return a / b * scale

df["share_0_14"] = safe_div(df["poblacion_0_14"], df["poblacion_total"], 100)
df["share_15_64"] = safe_div(df["poblacion_15_64"], df["poblacion_total"], 100)
df["share_65_mas"] = safe_div(df["poblacion_65_mas"], df["poblacion_total"], 100)
df["dependency_ratio"] = safe_div(df["poblacion_0_14"] + df["poblacion_65_mas"], df["poblacion_15_64"], 100)
df["densidad_viviendas_km2"] = safe_div(df["total_viviendas"], df["area_km2"])
df["delitos_por_100_negocios"] = df["ratio_delito_por_negocio"] * 100

sample = df[df["poblacion_total"] >= MIN_POPULATION].copy()
print(f"Analytic sample: {len(sample)} of {len(df)} AGEBs (population ≥ {MIN_POPULATION}).")

## 4. Distributions — choosing Pearson vs Spearman

Pearson assumes roughly linear, normally distributed variables without strong outliers.
Densities and per-capita rates are usually right-skewed, so we check skewness and normality
(D'Agostino–Pearson test) before deciding which coefficient is primary.

In [ ]:
VARIABLES = {
    "densidad_poblacion_km2": "Population density (res/km²)",
    "tasa_pea_porcentaje": "EAP rate (%)",
    "share_65_mas": "Population 65+ (%)",
    "dependency_ratio": "Dependency ratio",
    "densidad_viviendas_km2": "Housing density (dwellings/km²)",
    "densidad_negocios_km2": "Business density (est/km²)",
    "negocios_por_mil_hab": "Businesses per 1,000 residents",
    "densidad_comercio_km2": "Retail density (est/km²)",
    "densidad_servicios_km2": "Service density (est/km²)",
    "tasa_delictiva_por_mil_hab": "Crime rate (per 1,000 res.)",
    "delitos_por_100_negocios": "Crime per 100 establishments",
}
AGE_VARS = {"share_65_mas", "dependency_ratio"}
CRIME_VARS = {"tasa_delictiva_por_mil_hab", "delitos_por_100_negocios"}
BUSINESS_VARS = {"densidad_negocios_km2", "negocios_por_mil_hab", "densidad_comercio_km2", "densidad_servicios_km2"}

active_vars = [v for v in VARIABLES
               if not (v in AGE_VARS and not AGE_VALID)
               and not (v in CRIME_VARS and not CRIME_LOADED)
               and not (v in BUSINESS_VARS and not BUSINESS_LOADED)
               and sample[v].notna().sum() > 10 and sample[v].nunique() > 2]

rows = []
for v in active_vars:
    x = sample[v].dropna()
    k2, p_norm = stats.normaltest(x) if len(x) >= 20 else (np.nan, np.nan)
    rows.append({"variable": VARIABLES[v], "n": len(x), "skewness": stats.skew(x),
                 "skew_log1p": stats.skew(np.log1p(x.clip(lower=0))), "normality_p": p_norm})
dist = pd.DataFrame(rows).set_index("variable").round(3)
dist

In [ ]:
n = len(active_vars)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.1 * nrows))
for ax, v in zip(np.ravel(axes), active_vars):
    sns.histplot(sample[v].dropna(), bins=40, ax=ax, color="#2a78b8", edgecolor="white")
    ax.set_title(VARIABLES[v], fontsize=10)
    ax.set_xlabel("")
for ax in np.ravel(axes)[n:]:
    ax.set_visible(False)
fig.suptitle("Distribution of AGEB-level KPIs (analytic sample)", y=1.01, fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "dist_kpis_histograms.png")
plt.show()

**Decision.** Most density and rate variables are strongly right-skewed and fail the normality
test, so **Spearman's ρ is the primary coefficient** (rank-based, robust to outliers, captures
monotonic non-linear relationships). Pearson's *r* is reported on `log1p`-transformed values as a
secondary check of linear association.

## 5. Correlation matrices

In [ ]:
corr_df = sample[active_vars]
spearman = corr_df.corr(method="spearman")
pearson_log = np.log1p(corr_df.clip(lower=0)).corr(method="pearson")
labels = [VARIABLES[v] for v in active_vars]


def plot_corr(mat, title, fname):
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    sns.set_style("white")
    fig, ax = plt.subplots(figsize=(1.0 * len(mat) + 3, 0.85 * len(mat) + 2))
    sns.heatmap(mat, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.7, "label": "coefficient"},
                xticklabels=labels, yticklabels=labels, ax=ax, annot_kws={"size": 8})
    ax.set_title(title, fontsize=13, pad=12)
    plt.xticks(rotation=40, ha="right")
    fig.savefig(FIG_DIR / fname)
    plt.show()
    sns.set_style("whitegrid")


plot_corr(spearman, "Spearman ρ — AGEB KPIs, Mérida", "corr_matrix_spearman.png")
plot_corr(pearson_log, "Pearson r (log1p) — AGEB KPIs, Mérida", "corr_matrix_pearson_log.png")

## 6. Hypothesis tests (≥ 3 relationships)

| # | Relationship | Expected sign |
| :-: | :--- | :-: |
| H1 | Population density ↔ Business density | + |
| H2 | EAP rate ↔ Businesses per 1,000 residents | + |
| H3 | Population 65+ (%) ↔ Service density | + |
| H4 | Business density ↔ Crime rate | + |
| H5 | Retail density ↔ Crime per 100 establishments | ? |

p-values are adjusted for multiple testing with **Holm's method**. Strength labels follow
|ρ| < 0.1 negligible, < 0.3 weak, < 0.5 moderate, ≥ 0.5 strong.

In [ ]:
HYPOTHESES = [
    ("H1", "densidad_poblacion_km2", "densidad_negocios_km2", "+"),
    ("H2", "tasa_pea_porcentaje", "negocios_por_mil_hab", "+"),
    ("H3", "share_65_mas", "densidad_servicios_km2", "+"),
    ("H4", "densidad_negocios_km2", "tasa_delictiva_por_mil_hab", "+"),
    ("H5", "densidad_comercio_km2", "delitos_por_100_negocios", "?"),
]


def strength(r):
    a = abs(r)
    return "negligible" if a < 0.1 else "weak" if a < 0.3 else "moderate" if a < 0.5 else "strong"


def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj


results = []
for hid, x, y, expected in HYPOTHESES:
    if x not in active_vars or y not in active_vars:
        print(f"{hid}: skipped ({x} or {y} not available in the warehouse yet)")
        continue
    pair = sample[["cvegeo", x, y]].dropna()
    rho, p_s = stats.spearmanr(pair[x], pair[y])
    r, p_p = stats.pearsonr(np.log1p(pair[x].clip(lower=0)), np.log1p(pair[y].clip(lower=0)))
    results.append({"id": hid, "x": x, "y": y, "x_label": VARIABLES[x], "y_label": VARIABLES[y],
                    "expected_sign": expected, "n": len(pair),
                    "spearman_rho": rho, "spearman_p": p_s, "pearson_r_log": r, "pearson_p": p_p})

res = pd.DataFrame(results)
if not res.empty:
    res["spearman_p_holm"] = holm(res["spearman_p"])
    res["significant"] = res["spearman_p_holm"] < ALPHA
    res["strength"] = res["spearman_rho"].apply(strength)
    res["direction"] = np.where(res["spearman_rho"] >= 0, "positive", "negative")
    res.to_csv(ANALYTICS_DIR / "correlation_results.csv", index=False)
    display_cols = ["id", "x_label", "y_label", "n", "spearman_rho", "spearman_p_holm",
                    "pearson_r_log", "pearson_p", "strength", "direction", "significant"]
    display(res[display_cols].round(4))

In [ ]:
if not res.empty:
    PERCENT_VARS = {"tasa_pea_porcentaje", "share_65_mas", "dependency_ratio"}
    k = len(res)
    ncols = min(k, 3)
    nrows = int(np.ceil(k / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.3 * nrows), squeeze=False)
    for ax, (_, row) in zip(axes.ravel(), res.iterrows()):
        pair = sample[[row["x"], row["y"]]].dropna()
        ax.scatter(pair[row["x"]], pair[row["y"]], s=12, alpha=0.55, color="#2a78b8", edgecolor="none")
        # Log axes only for skewed density/rate variables; percentages stay linear
        for var, setter in ((row["x"], ax.set_xscale), (row["y"], ax.set_yscale)):
            if var not in PERCENT_VARS and (pair[var] > 0).all():
                setter("log")
        ax.set_xlabel(row["x_label"], fontsize=9)
        ax.set_ylabel(row["y_label"], fontsize=9)
        sig = "✓" if row["significant"] else "n.s."
        ax.set_title(f"{row['id']}: ρ = {row['spearman_rho']:.2f} ({sig}), n = {row['n']}", fontsize=10)
    for ax in axes.ravel()[k:]:
        ax.set_visible(False)
    fig.suptitle("Tested relationships (log axes for densities and rates)", y=1.02, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "corr_hypotheses_scatter.png")
    plt.show()

In [ ]:
# Automatic plain-language summary for the technical report
for _, row in res.iterrows():
    verdict = "statistically significant" if row["significant"] else "not statistically significant"
    print(f"{row['id']} — {row['x_label']} vs {row['y_label']}: {row['strength']} {row['direction']} "
          f"association (Spearman ρ = {row['spearman_rho']:.3f}, Holm-adjusted p = {row['spearman_p_holm']:.2g}, "
          f"n = {row['n']}); {verdict} at α = {ALPHA}. "
          f"Pearson r on log values = {row['pearson_r_log']:.3f}.")

## 7. Categorical breakdowns (sectors and crime by type/time)

In [ ]:
SCIAN_SECTORS = {
    "11": "Agriculture", "21": "Mining", "22": "Utilities", "23": "Construction",
    "31": "Manufacturing", "32": "Manufacturing", "33": "Manufacturing",
    "43": "Wholesale trade", "46": "Retail trade", "48": "Transportation", "49": "Postal & storage",
    "51": "Information", "52": "Finance & insurance", "53": "Real estate", "54": "Professional services",
    "55": "Corporate management", "56": "Business support", "61": "Education", "62": "Health care",
    "71": "Recreation & culture", "72": "Lodging & food", "81": "Other services", "93": "Government",
}
if not df_sector.empty:
    df_sector["sector_nombre"] = df_sector["sector_codigo"].astype(str).map(SCIAN_SECTORS).fillna("Other")
    sectors = (df_sector.groupby(["sector_nombre"], as_index=False)
               .agg(establecimientos=("establecimientos", "sum"),
                    categoria_macro=("categoria_macro", "first"),
                    sector_codigo=("sector_codigo", lambda c: "-".join(sorted({min(c), max(c)}))))
               .sort_values("establecimientos", ascending=False))
    display(sectors.head(12))
else:
    sectors = pd.DataFrame()
    print("No sector data (offline fallback or empty fact_negocios).")

if CRIME_LOADED and not df_crime_t.empty:
    crime_by_cat = df_crime_t.groupby("categoria_delito")["incidentes"].sum().sort_values(ascending=False)
    display(crime_by_cat.to_frame())

## 8. Export data feed for the dashboard (ApexCharts)

In [ ]:
def clean(v):
    """Convert numpy/pandas values to JSON-safe Python values."""
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return None if not np.isfinite(v) else round(float(v), 4)
    if isinstance(v, (np.bool_,)):
        return bool(v)
    return v


tot_pop = df["poblacion_total"].sum()
tot_area = df["area_km2"].sum()
tot_bus = df["total_negocios"].sum()
tot_crime = df["total_delitos"].sum()
city = {
    "agebs": len(df),
    "poblacion_total": tot_pop,
    "area_km2": tot_area,
    "densidad_poblacion_km2": tot_pop / tot_area,
    "tasa_pea_porcentaje": (df["poblacion_pea"].sum() / (df["poblacion_pea"].sum() + df["poblacion_pnea"].sum()) * 100)
    if AGE_VALID else None,
    "total_negocios": tot_bus,
    "densidad_negocios_km2": tot_bus / tot_area,
    "negocios_por_mil_hab": tot_bus / tot_pop * 1000 if tot_pop else None,
    "total_delitos": tot_crime if CRIME_LOADED else None,
    "tasa_delictiva_por_mil_hab": tot_crime / tot_pop * 1000 if CRIME_LOADED else None,
    "delitos_por_100_negocios": tot_crime / tot_bus * 100 if CRIME_LOADED and tot_bus else None,
}

summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": DATA_SOURCE,
    "min_population_rule": MIN_POPULATION,
    "flags": {"age_groups_valid": AGE_VALID, "business_layer_loaded": BUSINESS_LOADED,
              "crime_layer_loaded": CRIME_LOADED},
    "city": {k: clean(v) for k, v in city.items()},
    "age_groups": [
        {"group": "0-14", "value": clean(df["poblacion_0_14"].sum())},
        {"group": "15-64", "value": clean(df["poblacion_15_64"].sum())},
        {"group": "65+", "value": clean(df["poblacion_65_mas"].sum())},
    ] if AGE_VALID else [],
    "sectors": [{"code": str(r.sector_codigo), "name": r.sector_nombre, "macro": r.categoria_macro,
                 "count": clean(r.establecimientos)} for r in sectors.itertuples()] if not sectors.empty else [],
    "macro_categories": (df_sector.groupby("categoria_macro")["establecimientos"].sum()
                         .sort_values(ascending=False).reset_index()
                         .rename(columns={"categoria_macro": "name", "establecimientos": "count"})
                         .assign(count=lambda d: d["count"].astype(int)).to_dict("records"))
                        if not df_sector.empty else [],
    "correlation_matrix": {
        "variables": [{"key": v, "label": VARIABLES[v]} for v in active_vars],
        "spearman": [[clean(x) for x in row] for row in spearman.values],
        "pearson_log": [[clean(x) for x in row] for row in pearson_log.values],
    },
    "relationships": [],
    "crime": {},
    "agebs": [],
}

for _, row in res.iterrows():
    pts = sample[["cvegeo", row["x"], row["y"]]].dropna()
    summary["relationships"].append({
        **{k: clean(row[k]) for k in ("id", "x", "y", "x_label", "y_label", "expected_sign", "n",
                                       "spearman_rho", "spearman_p", "spearman_p_holm",
                                       "pearson_r_log", "pearson_p", "strength", "direction", "significant")},
        "points": [[clean(a), clean(b), c] for c, a, b in pts.itertuples(index=False)],
    })

if CRIME_LOADED and not df_crime_t.empty:
    ct = df_crime_t.copy()
    period_order = ["Morning", "Afternoon", "Evening", "Night", "Unknown"]
    periods = [p for p in period_order if p in set(ct["periodo_dia"])] or sorted(ct["periodo_dia"].unique())
    days = [d for d in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday",
                        "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
            if d in set(ct["dia_semana"].dropna())]
    heat = ct.pivot_table(index="periodo_dia", columns="dia_semana", values="incidentes", aggfunc="sum", fill_value=0)
    monthly = (ct.dropna(subset=["anio", "mes"]).assign(period=lambda d: d["anio"].astype(int).astype(str) + "-" +
                                                         d["mes"].astype(int).astype(str).str.zfill(2))
               .groupby(["period", "categoria_delito"])["incidentes"].sum().unstack(fill_value=0).sort_index())
    summary["crime"] = {
        "by_category": [{"category": k, "count": clean(v)} for k, v in
                        ct.groupby("categoria_delito")["incidentes"].sum().sort_values(ascending=False).items()],
        "period_by_day": {"periods": periods, "days": days,
                          "values": [[clean(heat.at[p, d]) if (p in heat.index and d in heat.columns) else 0
                                      for d in days] for p in periods]},
        "monthly": {"periods": list(monthly.index),
                    "series": [{"name": c, "data": [clean(x) for x in monthly[c].values]} for c in monthly.columns]},
    }

AGEB_FIELDS = ["cvegeo", "poblacion_total", "densidad_poblacion_km2", "tasa_pea_porcentaje",
               "poblacion_0_14", "poblacion_15_64", "poblacion_65_mas", "total_negocios",
               "densidad_negocios_km2", "negocios_por_mil_hab", "densidad_comercio_km2",
               "densidad_servicios_km2", "actividad_economica_dominante", "total_delitos",
               "tasa_delictiva_por_mil_hab", "delitos_por_100_negocios"]
summary["agebs"] = [{k: clean(v) for k, v in rec.items()} for rec in df[AGEB_FIELDS].to_dict("records")]

payload = json.dumps(summary, ensure_ascii=False, default=clean)
(ANALYTICS_DIR / "analytics_summary.json").write_text(payload, encoding="utf-8")
(FRONTEND_DATA_DIR / "analytics_summary.json").write_text(payload, encoding="utf-8")
print(f"✅ analytics_summary.json written ({len(payload) / 1024:.0f} KB) to outputs/analytics and src/frontend/public/data")

## 9. Cross-validation against the warehouse

The dashboard must show the same numbers as the SQL views. These checks recompute a few KPIs in
pandas from the base counts and compare them with the values returned by `v_kpis_territoriales`.

In [ ]:
checks = {
    "densidad_poblacion_km2": safe_div(df["poblacion_total"], df["area_km2"]),
    "densidad_negocios_km2": safe_div(df["total_negocios"], df["area_km2"]),
    "negocios_por_mil_hab": safe_div(df["total_negocios"], df["poblacion_total"], 1000),
}
if CRIME_LOADED:
    checks["tasa_delictiva_por_mil_hab"] = safe_div(df["total_delitos"], df["poblacion_total"], 1000)
for col, recomputed in checks.items():
    diff = (df[col] - recomputed).abs()
    ok = bool((diff.fillna(0) <= 0.01 + 1e-9).all() or diff.dropna().max() <= 0.011)
    print(f"{'✅' if ok else '❌'} {col:30s} max |view − recomputed| = {diff.max():.4f}")

## 10. Interpretation cautions

* **Association ≠ causation.** A positive ρ between business density and crime rate does not mean
  businesses cause crime; both may respond to a third factor (central location, foot traffic).
* **Spatial autocorrelation.** Neighbouring AGEBs are not independent observations, so classical
  p-values are optimistic. The Global/Local/Bivariate Moran's I analysis (backend module) is the
  spatially-aware complement to these results.
* **MAUP.** Results depend on the AGEB as unit of analysis; very small polygons inflate densities.
* **Per-capita denominators.** Residential population is the denominator, but commercial areas
  (e.g. Centro) receive many non-resident visitors, inflating per-1,000-resident rates.
* **Temporal mismatch.** Census (2020), DENUE (latest release) and crime records cover different
  dates; the analysis is cross-sectional.